In [ ]:
pip install openai

In [ ]:
import openai
import pandas as pd

In [ ]:
client = openai.OpenAI(api_key="#api_key")

In [ ]:
df = pd.read_excel("#raw_dataset_file")

In [ ]:
def classify_review(review_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content":
              """
              Classify the following ride-hailing app review as either 'Subjective' or 'Objective' based on these definitions:

              **Subjective:** The review contains opinions, emotions, imprecise terms, generalizations, expectations, or subjective comparisons. It may express personal dissatisfaction, general frustration, or complaints without clear evidence of a system-wide issue or policy violation.

              **Objective:** The review presents **verifiable facts**, describes **system behaviors**, reports **technical issues**, or documents **specific events**. This includes **driver misconduct** (e.g., overcharging, refusing service, unfair cancellations), **app malfunctions**, or **quantifiable data** (e.g., incorrect fare, excessive wait times caused by system errors).

              ---

              ### **Step-by-Step Chain of Thought Classification:**
              1. **Identify key phrases** that indicate subjectivity (opinions, emotions) or objectivity (facts, numbers, system behaviors).

              2. **Check for system-related failures and driver misconduct**:
                - If the review reports an **app malfunction, incorrect response, missing feature, fare miscalculation, GPS failure, or a verifiable technical issue**, classify it as **Objective**.
                - If the review **accuses a driver of misconduct** (e.g., overcharging, refusing service, threatening behavior, canceling unfairly), classify it as **Objective**, but **only if there is a clear pattern or verifiable context**.
                - If the complaint uses **vague frequency terms** (e.g., *"suka," "sering," "kadang-kadang"*) without measurable data, classify it as **Subjective**.
                - **Examples:**
                  - *"Harga yang argo di penumpang sama si sopir beda."* → **Objective** (Fare discrepancy).
                  - *"Driver maksa buat cancel padahal udah di lokasi."* → **Objective** (Driver misconduct).
                  - *"Driver suka cancel tiba-tiba."* → **Subjective** (Vague frequency, no proof of repeated misconduct).

              3. **Analyze measurable details**:
                - If the review contains **quantifiable data** (e.g., 'waiting for 1 hour', 'fare increased by 20%'), check the context:
                  - If it describes a **technical issue, app malfunction, or verifiable driver misconduct**, classify it as **Objective**.
                  - If it expresses frustration, disappointment, or an expectation **without a clear system failure**, classify it as **Subjective**.
                - **Examples:**
                  - *"Sudah nunggu 1 jam tapi driver tidak ada."*
                    - If due to **app error (e.g., driver keeps disappearing)** → **Objective**.
                    - If due to **traffic conditions or bad luck** → **Subjective**.

              4. **Determine if the issue is personal or systemic**:
                - If the complaint describes **a systemic failure (e.g., GPS wrong, payment not processed, app crash, incorrect charges)** → **Objective**.
                - If it is **personal dissatisfaction (e.g., driver rude, expensive fares, long wait, general frustration)** → **Subjective**.
                - If a term like *"jauh," "lama," "mahal"* is used **without specific numbers**, classify it as **Subjective**.
                - **Examples:**
                  - *"Gojek makin mahal aja, dikira semua orang kaya?"* → **Subjective** (General complaint, no proof of system issue).
                  - *"Dapat driver jauh terus."* → **Subjective** (No specific distance).
                  - *"Dapat driver 15 km padahal biasanya 5 km."* → **Objective** (Measurable comparison).

              5. **Final Decision**:
                - Respond with only one word: 'Subjective' or 'Objective'.
                """

                                         },
            {"role": "user", "content": f"Classify this sentence: \"{review_text}\""}
        ],
        temperature=0.1,
        top_p =0.8,
        max_tokens=10
    )

    return response.choices[0].message.content.strip()

In [ ]:
df["classification"] = df["cleaned_review"].apply(classify_review)

In [ ]:
df.to_excel("#labeled_dataset_file",, index=False)